# 📊 Monitoring MLOps, Santé Opérationnelle & Détection du Data Drift

Ce notebook regroupe l'ensemble du workflow MLOps de production :
1. **Inspection des dépendances et chargement des artefacts** (modèle XGBoost/Pipeline)
2. **Analyse de la santé opérationnelle de l'API** (volume, taux de succès, latence P95)
3. **Détection du Data Drift & Target Drift** via le framework **Evidently AI**
4. **Diagnostic MLOps et plan d'action**

## 1. Environment & Imports

In [ ]:
import json
import os
from pathlib import Path
import joblib
import pandas as pd
from IPython.display import HTML, display

from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset
from evidently.pipeline.column_mapping import ColumnMapping

# Définition de la racine du projet (s'adapte si exécuté depuis /notebooks ou la racine)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Chemins d'accès aux fichiers et rapports
LOGS_FILE = PROJECT_ROOT / "logs" / "predictions.jsonl"
REFERENCE_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "X_train.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
REPORT_HTML_PATH = REPORTS_DIR / "data_drift_report.html"

MODEL_CANDIDATE_PATHS = [
    PROJECT_ROOT / "artifacts" / "best_model_pipeline.joblib",
    PROJECT_ROOT / "models" / "best_pipeline_xgboost_epure.pkl",
    PROJECT_ROOT / "models" / "best_pipeline_production.joblib"
]

# Variables du modèle
NUMERICAL_FEATURES = [
    "customer_value_score",
    "Panier_Moyen_N_signature_3",
    "annees_depuis_dernier_achat",
    "Turnover_N_signature_1",
    "Panier_Moyen_N_signature_1",
    "%EC",
    "Nb_lignes_N_signature_1",
    "Turnover_N_signature_3",
    "Famille_2_N_signature_2",
    "Panier_Moyen_N_signature_2",
    "annees_depuis_1ere_facture",
    "Famille_0_N_signature_1",
    "Famille_2_N_signature_1",
    "Famille_11_N_signature_1",
    "Famille_14_N_signature_1",
    "Famille_9_N_signature_3",
]

CATEGORICAL_FEATURES = [
    "GrandCompte",
    "act_val_cust_3M",
    "clp_contrat_ap_stat",
    "division",
]

FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
print(f"✅ Configuration initialisée. Project Root: {PROJECT_ROOT}")

## 2. Chargement du Modèle & Extraction des Artefacts

In [ ]:
def load_model():
    """Recherche et charge le pipeline entraîné."""
    for model_path in MODEL_CANDIDATE_PATHS:
        if model_path.exists():
            print(f"📍 Chargement du modèle depuis : {model_path}")
            return joblib.load(model_path)
    print("⚠️ Aucun artefact de modèle trouvé dans les chemins spécifiés.")
    return None

pipeline = load_model()
if pipeline is not None:
    print("✅ Pipeline de modèle chargé avec succès !")

## 3. Fonctions Utilitaires

In [ ]:
def load_production_logs(log_path: Path) -> pd.DataFrame:
    """Lit le fichier predictions.jsonl et extrait les inputs ainsi que la prédiction."""
    if not log_path.exists():
        raise FileNotFoundError(f"Le fichier de log {log_path} n'existe pas.")

    records = []
    with open(log_path, mode="r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                if data.get("status") == "success":
                    entry = data["inputs"].copy()
                    entry["prediction"] = data.get("prediction")
                    records.append(entry)

    return pd.DataFrame(records)

def ensure_reference_predictions(reference_df: pd.DataFrame) -> pd.DataFrame:
    """Calcule les prédictions sur le jeu de référence si la colonne 'prediction' manque."""
    if "prediction" in reference_df.columns:
        return reference_df

    if pipeline is None:
        return reference_df

    df = reference_df.copy()
    for col in CATEGORICAL_FEATURES:
        if col in df.columns:
            df[col] = df[col].astype("category")

    cols_for_model = [c for c in df.columns if c in FEATURES]

    expected_order = getattr(
        getattr(pipeline, "named_steps", {}).get("imputer"), "feature_names_in_", None
    )
    if expected_order is not None:
        cols_for_model = list(expected_order)

    df["prediction"] = pipeline.predict(df[cols_for_model])
    return df

## 4. Analyse de la Santé Opérationnelle de l'API

In [ ]:
if LOGS_FILE.exists():
    df_raw = []
    with open(LOGS_FILE, mode="r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                df_raw.append(json.loads(line))
    
    if df_raw:
        df_logs = pd.DataFrame(df_raw)
        total_reqs = len(df_logs)
        success_rate = (df_logs["status"] == "success").mean() * 100
        
        print("========================================")
        print("📈 MÉTRIQUES OPÉRATIONNELLES DE L'API")
        print(f" • Volume total de requêtes : {total_reqs}")
        print(f" • Taux de succès           : {success_rate:.2f}%")
        
        if "latency_ms" in df_logs.columns:
            mean_lat = df_logs["latency_ms"].mean()
            p95_lat = df_logs["latency_ms"].quantile(0.95)
            print(f" • Latence moyenne         : {mean_lat:.2f} ms")
            print(f" • Latence P95             : {p95_lat:.2f} ms")
        print("========================================")
else:
    print(f"⚠️ Aucun fichier de log trouvé à : {LOGS_FILE}")

## 5. Détection du Data Drift & Target Drift (Evidently AI)

In [ ]:
print("🔄 Chargement et préparation des jeux de données...")

current_df = load_production_logs(LOGS_FILE)
print(f"📊 Logs de production chargés : {len(current_df)} enregistrements.")

if REFERENCE_DATA_PATH.exists():
    reference_df = pd.read_csv(REFERENCE_DATA_PATH)
    reference_df = ensure_reference_predictions(reference_df)
else:
    print(f"⚠️ Fichier de référence introuvable ({REFERENCE_DATA_PATH}). Simulation par découpage des logs.")
    split_idx = int(len(current_df) * 0.5)
    reference_df = current_df.iloc[:split_idx].copy()
    current_df = current_df.iloc[split_idx:].copy()

cols_to_compare = [c for c in FEATURES if c in current_df.columns and c in reference_df.columns]

column_mapping = ColumnMapping()
column_mapping.numerical_features = [c for c in NUMERICAL_FEATURES if c in cols_to_compare]
column_mapping.categorical_features = [c for c in CATEGORICAL_FEATURES if c in cols_to_compare]

has_prediction = "prediction" in current_df.columns and "prediction" in reference_df.columns
if has_prediction:
    cols_to_compare.append("prediction")
    column_mapping.prediction = "prediction"

ref_subset = reference_df[cols_to_compare]
curr_subset = current_df[cols_to_compare]

# Génération du rapport Evidently
metrics = [DataDriftPreset()]
if has_prediction:
    metrics.append(TargetDriftPreset())

drift_report = Report(metrics=metrics)
drift_report.run(reference_data=ref_subset, current_data=curr_subset, column_mapping=column_mapping)

# Sauvegarde HTML
drift_report.save_html(str(REPORT_HTML_PATH))
print(f"✅ Rapport Data Drift sauvegardé dans : {REPORT_HTML_PATH}")

# Affichage HTML interactif
HTML(filename=str(REPORT_HTML_PATH))

## 6. Synthèse MLOps & Diagnostic de Dérive

### Bilan Diagnostic
* **Covariate Shift (Data Drift) :** Dérive détectée sur les variables de volume d'achat et de chiffre d'affaires (`Panier_Moyen_N_signature_1`, `Turnover_N_signature_1`, etc.).
* **Prediction Drift (Target Drift) :** Modification sensible des scores prédits $\hat{Y}$ en sortie de modèle.

### Stratégie MLOps de Réponse au Drift
1. **Phase 1 - Diagnostic Qualité :** Vérification de l'intégrité du pipeline ETL et absence d'anomalies de formatage.
2. **Phase 2 - Qualification Métier :** Analyse des évolutions comportementales ou de la saisonnalité.
3. **Phase 3 - Ré-entraînement Automatisé :** Application d'un fenêtrage glissant pour ré-entraîner XGBoost et re-validation relative de la performance ($F_1$-score / AUC) avant mise en production.